In [ ]:
# Cell 1 — Imports and load dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

df = pd.read_csv("../data/labels_clean.csv")
print(f"Loaded {len(df):,} rows")
df.head()

In [ ]:
# Cell 2 — Shape, dtypes, null counts
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())
print("\nSample rows:")
df.sample(5)

In [ ]:
# Cell 3 — Vertical distribution
vc = df["vertical"].value_counts()
fig, ax = plt.subplots(figsize=(7, 4))
vc.plot(kind="bar", ax=ax, color=["#5c1c87", "#dc2626", "#0f172a", "#64748b"])
ax.set_title("Ads per vertical", fontsize=14)
ax.set_xlabel("")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=0)
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}",
                (p.get_x() + p.get_width() / 2, p.get_height()),
                ha="center", va="bottom", fontsize=11)
plt.tight_layout()
plt.show()
print(vc)

In [ ]:
# Cell 4 — Source distribution
sc = df["source"].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sc.plot(kind="bar", ax=axes[0], color=["#1e40af", "#059669"])
axes[0].set_title("Ads by source")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height()):,}",
                     (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha="center", va="bottom")

# Censored vs non-censored breakdown
cens = df["censored"].value_counts()
cens.index = ["censored" if x else "observed" for x in cens.index]
cens.plot(kind="pie", ax=axes[1], autopct="%1.1f%%",
          colors=["#f59e0b", "#3b82f6"], startangle=90)
axes[1].set_title("Censored vs observed halflife")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()
print(sc)

In [ ]:
# Cell 5 — halflife_days histogram (observed only, censored excluded)
observed = df[df["censored"] == False]["halflife_days"].dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(observed, bins=50, color="#3b82f6", edgecolor="white", linewidth=0.3)
axes[0].set_title(f"halflife_days distribution (n={len(observed):,})")
axes[0].set_xlabel("Days")
axes[0].set_ylabel("Count")
axes[0].axvline(observed.median(), color="#ef4444", linestyle="--",
                label=f"median={observed.median():.1f}d")
axes[0].legend()

# Per-vertical
for v, color in zip(["gaming", "ecommerce", "finance"],
                    ["#5c1c87", "#dc2626", "#0f172a"]):
    sub = df[(df["vertical"] == v) & (df["censored"] == False)]["halflife_days"].dropna()
    axes[1].hist(sub, bins=40, alpha=0.6, label=v, color=color)
axes[1].set_title("halflife_days by vertical (observed)")
axes[1].set_xlabel("Days")
axes[1].legend()

plt.tight_layout()
plt.show()
print(observed.describe())

In [ ]:
# Cell 6 — ctr_score histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df["ctr_score"], bins=50, color="#10b981", edgecolor="white", linewidth=0.3)
axes[0].set_title(f"ctr_score distribution (n={len(df):,})")
axes[0].set_xlabel("CTR score (0–1)")
axes[0].set_ylabel("Count")
axes[0].axvline(df["ctr_score"].median(), color="#ef4444", linestyle="--",
                label=f"median={df['ctr_score'].median():.3f}")
axes[0].legend()

for v, color in zip(["gaming", "ecommerce", "finance"],
                    ["#5c1c87", "#dc2626", "#0f172a"]):
    sub = df[df["vertical"] == v]["ctr_score"]
    axes[1].hist(sub, bins=40, alpha=0.6, label=v, color=color)
axes[1].set_title("ctr_score by vertical")
axes[1].set_xlabel("CTR score")
axes[1].legend()

plt.tight_layout()
plt.show()
print(df["ctr_score"].describe())

In [ ]:
# Cell 7 — ctr vs halflife scatter + sample image grid

# --- Part A: scatter plot ---
observed_df = df[df["censored"] == False].dropna(subset=["halflife_days"])
colors = {"gaming": "#5c1c87", "ecommerce": "#dc2626",
          "finance": "#0f172a", "other": "#94a3b8"}

fig, ax = plt.subplots(figsize=(9, 5))
for v, grp in observed_df.groupby("vertical"):
    ax.scatter(grp["halflife_days"], grp["ctr_score"],
               alpha=0.25, s=8, color=colors.get(v, "#94a3b8"), label=v)
ax.set_xlabel("halflife_days (observed)")
ax.set_ylabel("ctr_score")
ax.set_title("CTR score vs Halflife — colored by vertical")
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

# --- Part B: sample image grid (9 images, 3 per vertical) ---
VERTICALS = ["gaming", "ecommerce", "finance"]
SAMPLES_PER_VERTICAL = 3

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
fig.suptitle("Sample ads — 3 per vertical (mix of apify + synthetic)",
             fontsize=13, y=1.01)

for row_idx, vertical in enumerate(VERTICALS):
    pool = df[df["vertical"] == vertical].copy()
    # Try to get a mix of sources; fall back to any if not enough
    apify_pool = pool[pool["source"] == "apify"]
    synth_pool = pool[pool["source"] == "synthetic"]

    picked = []
    if len(apify_pool) >= 1:
        picked.append(apify_pool.sample(min(1, len(apify_pool))).iloc[0])
    if len(synth_pool) >= 2:
        for _, r in synth_pool.sample(min(2, len(synth_pool))).iterrows():
            picked.append(r)
    # Pad with whatever is available if we don't have 3
    if len(picked) < SAMPLES_PER_VERTICAL:
        remaining = pool[~pool["ad_id"].isin([p["ad_id"] for p in picked])]
        for _, r in remaining.sample(min(SAMPLES_PER_VERTICAL - len(picked),
                                         len(remaining))).iterrows():
            picked.append(r)

    for col_idx in range(SAMPLES_PER_VERTICAL):
        ax = axes[row_idx][col_idx]
        if col_idx >= len(picked):
            ax.axis("off")
            ax.set_title("no data", fontsize=8)
            continue
        row = picked[col_idx]
        img_path = Path("..") / row["image_path"]
        try:
            img = mpimg.imread(str(img_path))
            ax.imshow(img)
        except Exception:
            ax.set_facecolor("#e2e8f0")
            ax.text(0.5, 0.5, "image\nnot found", ha="center", va="center",
                    transform=ax.transAxes, fontsize=8, color="#64748b")
        halflife_str = f"{row['halflife_days']:.1f}d" if pd.notna(row["halflife_days"]) else "censored"
        ax.set_title(
            f"{vertical} | {row['source']}\nctr={row['ctr_score']:.3f}  hl={halflife_str}",
            fontsize=8
        )
        ax.axis("off")

plt.tight_layout()
plt.show()